In [1]:
import os
import torch

from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from yacs.config import CfgNode as CN

from data.dataset import make_dataset
from src.model import Model
from src.utils import clean_exp_savedir
from src.losses import supervised_loss
import argparse

In [2]:
source_train_loader, _, source_test_loader, target_test_loader = (
    make_dataset(
        source_dataset="office31_amazon",
        target_dataset="office31_dslr",
        img_size=384,
        train_bs=16,
        eval_bs=64,
        num_workers=16,
    )
)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DSSD_SignalExtractor(nn.Module):
    def __init__(self, num_radial_bins=32, num_angle_bins=18):
        super().__init__()
        self.num_radial_bins = num_radial_bins
        self.num_angle_bins = num_angle_bins
        
        # Fixed Sobel filters for Shape (Edge) Extraction
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    def extract_texture_fourier(self, x):
        """
        Extracts texture using 1D Radial Power Spectrum of the Fourier Transform.
        """
        # Convert to grayscale if RGB
        if x.shape[1] == 3:
            x = 0.2989 * x[:, 0:1] + 0.5870 * x[:, 1:2] + 0.1140 * x[:, 2:3]
            
        B, C, H, W = x.shape
        
        # 2D Fast Fourier Transform
        fft2d = torch.fft.fft2(x)
        fftshift = torch.fft.fftshift(fft2d)
        power_spectrum = torch.abs(fftshift) ** 2
        
        # Create radial distance map
        y, x_coord = torch.meshgrid(torch.arange(H), torch.arange(W), indexing='ij')
        center_y, center_x = H // 2, W // 2
        radius = torch.sqrt((y - center_y)**2 + (x_coord - center_x)**2).to(x.device)
        
        # Bin the power spectrum into radial rings (1D profile)
        max_radius = min(center_y, center_x)
        radial_profile = torch.zeros((B, self.num_radial_bins), device=x.device)
        
        bin_edges = torch.linspace(0, max_radius, self.num_radial_bins + 1, device=x.device)
        for i in range(self.num_radial_bins):
            mask = (radius >= bin_edges[i]) & (radius < bin_edges[i+1])
            # Sum power in this radial bin for each image in batch
            radial_profile[:, i] = power_spectrum[:, 0, mask].mean(dim=1)
            
        # Normalize to ensure stability
        radial_profile = F.normalize(radial_profile, p=2, dim=1)
        return radial_profile

    def extract_shape_gradients(self, x):
        """
        Extracts shape using a lightweight Histogram of Oriented Gradients (HOG) approximation.
        """
        if x.shape[1] == 3:
            x = 0.2989 * x[:, 0:1] + 0.5870 * x[:, 1:2] + 0.1140 * x[:, 2:3]
            
        # Compute image gradients
        grad_x = F.conv2d(x, self.sobel_x, padding=1)
        grad_y = F.conv2d(x, self.sobel_y, padding=1)
        
        # Compute magnitude and orientation
        magnitude = torch.sqrt(grad_x**2 + grad_y**2 + 1e-6)
        angle = torch.atan2(grad_y, grad_x) # Range: [-pi, pi]
        
        # Bin the angles into a histogram, weighted by magnitude
        B = x.shape[0]
        shape_profile = torch.zeros((B, self.num_angle_bins), device=x.device)
        angle_bins = torch.linspace(-torch.pi, torch.pi, self.num_angle_bins + 1, device=x.device)
        
        for i in range(self.num_angle_bins):
            mask = (angle >= angle_bins[i]) & (angle < angle_bins[i+1])
            # Sum the gradient magnitudes falling into this angle bin
            shape_profile[:, i] = (magnitude * mask).view(B, -1).sum(dim=1)
            
        shape_profile = F.normalize(shape_profile, p=2, dim=1)
        return shape_profile

In [4]:
import torch

def extract_all_signals(dataloader, extractor, device):
    """Passes an entire dataset through the backbone-free extractor."""
    extractor.eval()
    all_tex = []
    all_shape = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            
            tex_feat = extractor.extract_texture_fourier(images)
            shape_feat = extractor.extract_shape_gradients(images)
            
            all_tex.append(tex_feat)
            all_shape.append(shape_feat)
            all_labels.append(labels.to(device))
            
    return torch.cat(all_tex, dim=0), torch.cat(all_shape, dim=0), torch.cat(all_labels, dim=0)

def compute_classwise_source_distributions(features, labels, num_classes, device, epsilon=1e-5):
    """Calculates mu and the inverse covariance matrix for each class."""
    class_dists = {}
    
    for c in range(num_classes):
        # Isolate features for class c
        class_mask = (labels == c)
        class_feats = features[class_mask]
        
        # Handle cases where a class might be missing from the batch/dataset
        if class_feats.size(0) < 2:
            class_dists[c] = None
            continue
            
        # Calculate Mean
        mu = class_feats.mean(dim=0)
        
        # Calculate Covariance
        centered = class_feats - mu
        cov = (centered.T @ centered) / (class_feats.size(0) - 1)
        
        # Add epsilon to diagonal for numerical stability
        cov += torch.eye(cov.size(0), device=device) * epsilon
        
        # Pre-compute inverse covariance
        inv_cov = torch.linalg.inv(cov)
        
        class_dists[c] = {'mu': mu, 'inv_cov': inv_cov}
        
    return class_dists

def compute_mahalanobis_distance(x, mu, inv_cov):
    """Vectorized Mahalanobis distance for a batch of features."""
    delta = x - mu
    left_term = torch.matmul(delta, inv_cov)
    distance = torch.sqrt(torch.sum(left_term * delta, dim=1))
    return distance

def analyze_classwise_domain_gap(source_loader, target_loader, extractor, num_classes, device):
    """
    Measures the exact texture and shape shift for every class between two domains.
    """
    print("Extracting Source Signals...")
    src_tex, src_shape, src_labels = extract_all_signals(source_loader, extractor, device)
    
    print("Extracting Target Signals...")
    tgt_tex, tgt_shape, tgt_labels = extract_all_signals(target_loader, extractor, device)
    
    print("Building Class-wise Source Distributions...")
    src_tex_dists = compute_classwise_source_distributions(src_tex, src_labels, num_classes, device)
    src_shape_dists = compute_classwise_source_distributions(src_shape, src_labels, num_classes, device)
    
    class_texture_gaps = torch.zeros(num_classes, device=device)
    class_shape_gaps = torch.zeros(num_classes, device=device)
    
    print("Calculating Domain Shift per Class...\n")
    print(f"{'Class':<8} | {'Texture Gap':<15} | {'Shape Gap':<15} | {'Dominant Shift'}")
    print("-" * 60)
    
    for c in range(num_classes):
        # Skip if source didn't have enough samples to build a distribution
        if src_tex_dists[c] is None or src_shape_dists[c] is None:
            continue
            
        # Isolate target samples for class c
        tgt_mask = (tgt_labels == c)
        tgt_tex_c = tgt_tex[tgt_mask]
        tgt_shape_c = tgt_shape[tgt_mask]
        
        if tgt_tex_c.size(0) == 0:
            continue
            
        # Calculate how far target samples drifted from the source distribution
        tex_distances = compute_mahalanobis_distance(
            tgt_tex_c, src_tex_dists[c]['mu'], src_tex_dists[c]['inv_cov']
        )
        shape_distances = compute_mahalanobis_distance(
            tgt_shape_c, src_shape_dists[c]['mu'], src_shape_dists[c]['inv_cov']
        )
        
        # Average distance represents the total domain gap for this class
        avg_tex_gap = tex_distances.mean().item()
        avg_shape_gap = shape_distances.mean().item()
        
        class_texture_gaps[c] = avg_tex_gap
        class_shape_gaps[c] = avg_shape_gap
        
        dominant = "TEXTURE" if avg_tex_gap > avg_shape_gap else "SHAPE"
        
        print(f"{c:<8} | {avg_tex_gap:<15.4f} | {avg_shape_gap:<15.4f} | {dominant}")
        
    return class_texture_gaps, class_shape_gaps

In [7]:
num_classes = 31
extractor = DSSD_SignalExtractor(64, 8).to("cuda")
tex_gaps, shape_gaps = analyze_classwise_domain_gap(
    source_test_loader, target_test_loader, extractor, num_classes, 'cuda'
)

Extracting Source Signals...
Extracting Target Signals...
Building Class-wise Source Distributions...
Calculating Domain Shift per Class...

Class    | Texture Gap     | Shape Gap       | Dominant Shift
------------------------------------------------------------
0        | 10.9548         | 5.9641          | TEXTURE
1        | 6.8328          | 11.4597         | SHAPE
2        | 2.2707          | 4.5222          | SHAPE
3        | 2.8894          | 6.1891          | SHAPE
4        | 5.4114          | 4.4441          | TEXTURE
5        | 5.8761          | 7.2354          | SHAPE
6        | 5.2912          | 4.9030          | TEXTURE
7        | 8.2083          | 2.8032          | TEXTURE
8        | 4.4489          | 3.9779          | TEXTURE
9        | 3.0863          | 4.6943          | SHAPE
10       | 1.4697          | 6.1280          | SHAPE
11       | 4.5597          | 5.5944          | SHAPE
12       | 7.0514          | 4.6296          | TEXTURE
13       | 3.2658          | 2.9941